# Windblade YOLO11n train/validation apparatus
This output-free notebook runs the three frozen class-agnostic seeds. It uses only training and validation data; the held-out split remains sealed. Run cells in order on a Colab GPU runtime with at least 8 GiB VRAM.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/Stukhori/Bade-defect-recognition.git'
APPARATUS_COMMIT = 'e05bae9d786f43a7c6d8bf4b8a56ca5ee99e8b60'
REPOSITORY_ROOT = '/content/Bade-defect-recognition'
DRIVE_ROOT = '/content/drive/MyDrive/windblade_phase11b'
DATA_ROOT = '/content/windblade_phase11b_data'

Mounted at /content/drive


In [2]:
import pathlib, shutil, subprocess
if APPARATUS_COMMIT == 'REPLACE_WITH_APPARATUS_COMMIT':
    raise ValueError('Set the full apparatus commit before continuing')
if pathlib.Path(REPOSITORY_ROOT).exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, REPOSITORY_ROOT], check=True)
subprocess.run(['git', 'checkout', '--detach', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_ROOT, text=True).strip()
assert head == APPARATUS_COMMIT

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--requirement', f'{REPOSITORY_ROOT}/requirements-detection-colab.txt'], check=True)

In [ ]:
BASE = ['python', 'scripts/run_phase11b.py', '--drive-root', DRIVE_ROOT, '--data-root', DATA_ROOT]
subprocess.run(BASE + ['apparatus-check'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['verify-archive'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['preflight'], cwd=REPOSITORY_ROOT, check=True)

In [ ]:
record = pathlib.Path(DRIVE_ROOT) / 'provenance/phase11b_weight_acquisition.json'
if not record.exists():
    subprocess.run(BASE + ['acquire-weight', '--apparatus-commit', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
else:
    print('Using the existing immutable weight-acquisition record; training will verify its bytes.')

In [ ]:
subprocess.run(BASE + ['materialize-trainval'], cwd=REPOSITORY_ROOT, check=True)

In [ ]:
import os
for seed in (17, 29, 43):
    environment = dict(os.environ, PYTHONHASHSEED=str(seed), CUBLAS_WORKSPACE_CONFIG=':4096:8')
    subprocess.run(BASE + ['train', '--seed', str(seed)], cwd=REPOSITORY_ROOT, env=environment, check=True)

In [ ]:
subprocess.run(BASE + ['select-validation'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['bundle'], cwd=REPOSITORY_ROOT, check=True)

Stop here. Inspect the generated selection receipt, commit and push it from an authenticated checkout, and follow `docs/phase11b_colab.md` for the separately gated final evaluation. Do not change the frozen configuration or selections.